# Lewisham · `lpa_app_no` (web: "Application number")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Duplicate references

Brent had duplicates: 24 empty copies of full records, and 4 applications recorded under
both Brent and OPDC. Here: does any `lpa_app_no` appear on more than one Lewisham record,
and are the opinion-type decisions (No Objection / Objection Raised (OBS only), Comment
Issued, Closed) among them? `lpa_app_no` has no `.raw`, so every record is fetched and
counted in pandas; references are compared after trimming spaces and upper-casing.

In [2]:
COLS = ["id", "lpa_app_no", "lpa_name", "borough", "status", "decision", "decision_date", "bo_system", "site_name"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

refs = pd.DataFrame(rows).reindex(columns=COLS)
refs["ref"] = refs["lpa_app_no"].str.strip().str.upper()
OPINION = ["No Objection to Proposal (OBS only)", "Objection Raised to Proposal (OBS only)", "Comment Issued", "Closed"]
refs["opinion"] = refs["decision"].isin(OPINION)
dup = refs[refs.duplicated("ref", keep=False)].sort_values(["ref", "id"])
print("Records:", len(refs), "| distinct references:", refs["ref"].nunique(),
      "| records sharing a reference:", len(dup), "| references shared:", dup["ref"].nunique())
print("Opinion-type records:", int(refs["opinion"].sum()), "| of them sharing a reference:", int(dup["opinion"].sum()))
print("Same id twice:", int(refs["id"].duplicated().sum()))

Records: 29021 | distinct references: 29021 | records sharing a reference: 0 | references shared: 0
Opinion-type records: 431 | of them sharing a reference: 0
Same id twice: 0


In [3]:
pd.set_option("display.max_colwidth", 40)
dup.drop(columns=["ref", "opinion"]).head(40)

,id,lpa_app_no,lpa_name,borough,status,decision,decision_date,bo_system,site_name


### Findings · Step 1

- **No duplicates:** all 29,021 Lewisham records have a different `lpa_app_no` (after
  trimming and upper-casing), and no id appears twice. Unlike Brent, no reference is
  recorded twice.
- **The 431 opinion-type records** (No Objection / Objection Raised (OBS only), Comment
  Issued, Closed) are no exception: none shares a reference.
- **Not checkable here:** if these are Lewisham's comments on another borough's
  application, the original would be in that borough's data under that borough's own
  reference. Looking would need data from outside Lewisham (golden rule), so it is not
  done.

## Step 2 · Nulls and empty values

On the records fetched in Step 1: **null** (no value) and **empty or blank** (a string
with nothing in it, or only whitespace).

In [4]:
value = refs["lpa_app_no"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
summary = pd.Series({"null": int(is_null.sum()), "empty or blank": int(is_blank.sum()),
                     "filled": int((~is_null & ~is_blank).sum())})
summary["total"] = summary.sum()
summary.to_frame("count")

,count
null,0
empty or blank,0
filled,29021
total,29021


### Findings · Step 2

- **Every record has an `lpa_app_no`** (29,021 filled; no null, no blank). Brent was the
  same.

## Step 3 · Type check and formats

The Python type of every value. Then the shape of each reference: letters become `A`,
digits `9` (e.g. `DC/21/120320` → `AA/99/999999`), counted with an example and the
`bo_system`. Finally, whether `id` is always "Lewisham-" plus `lpa_app_no` with `/`
replaced by `_`.

In [5]:
import re

print("Types:", refs["lpa_app_no"].map(lambda v: type(v).__name__).value_counts().to_dict())
refs["shape"] = refs["lpa_app_no"].map(lambda v: re.sub(r"\d", "9", re.sub(r"[A-Za-z]", "A", v)))
shapes = refs.groupby("shape").agg(count=("id", "size"), example=("lpa_app_no", "first"),
                                   bo_system=("bo_system", lambda s: s.value_counts().to_dict()))
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 300)
shapes.sort_values("count", ascending=False).head(25)

Types: {'str': 29021}


,count,example,bo_system
shape,,,
AA/99/999999,24758,DC/10/075331,"{'CSV Data Import': 24521, 'FME Legacy Data Import': 210, 'Portal': 27}"
AA99/99999,2036,DC00/46204,"{'FME Legacy Data Import': 2033, 'Portal': 3}"
AA99/999999,823,DC06/063352,"{'FME Legacy Data Import': 820, 'Portal': 3}"
AA9999999,790,DC0045808,{'FME Legacy Data Import': 790}
AA99999,413,DC21155,{'FME Legacy Data Import': 413}
99/999999,106,19/111255,{'CSV Data Import': 106}
AAA/99/999999,41,PRE/18/107750,{'CSV Data Import': 41}
AA9999999A,16,DC0046436A,{'FME Legacy Data Import': 16}
AA99999A,9,DC30321A,{'FME Legacy Data Import': 9}


In [6]:
expected = "Lewisham-" + refs["lpa_app_no"].str.replace("/", "_")
print("id == 'Lewisham-' + lpa_app_no with / -> _:", int((refs["id"] == expected).sum()), "of", len(refs))
print(refs.loc[refs["id"] != expected, ["id", "lpa_app_no"]].head(10).to_string(index=False))
print("Distinct shapes:", refs["shape"].nunique())

id == 'Lewisham-' + lpa_app_no with / -> _: 29021 of 29021
Empty DataFrame
Columns: [id, lpa_app_no]
Index: []
Distinct shapes: 22


### Findings · Step 3

- **Every value is a string** (29,021); matches the schema's text type.
- **22 shapes, two families:**
  - **Current style `DC/yy/nnnnnn`** (24,758): the CSV import, plus 210 legacy and 27 Portal
    records. Also `PRE/yy/nnnnnn` (41), `TEST/yy/nnnnnn` (4, + 2 with `/X`), `yy/nnnnnn`
    without a prefix (106, CSV import) and one `DC/yy/nnnnnn/X` (DC/13/082322/X).
  - **Legacy styles** (FME Legacy Data Import): `DCyy/nnnnn` (2,036), `DCyy/nnnnnn` (823),
    `DCnnnnnnn` (790), `DCnnnnn` (413), and rare ones with letters (`DC0046436A`,
    `DCDE151ATP`, `DCLE240155TP`, …).
- **The year is not always readable:** `DC/21/…` and `DC06/…` show it; `DC0045808`,
  `DC21155` don't.
- **`id` is always "Lewisham-" + lpa_app_no with `/` → `_`** (29,021 of 29,021). So a
  reference given by a user can be turned into the id directly.
- References are not always the same length: `DC/13/83358` (5 digits) next to
  `DC/13/082322` (6).